<a href="https://colab.research.google.com/github/Raimbek-pro/ml-projects/blob/main/cnn_tsis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Split training set into train and validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

print(f"Training set: {len(train_dataset)} samples")
print(f"Validation set: {len(val_dataset)} samples")
print(f"Test set: {len(test_dataset)} samples")

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def visualize_dataset(dataloader, num_examples=10):
    dataiter = iter(dataloader)
    images, labels = next(dataiter)

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.ravel()

    for i in range(num_examples):
        # Remove channel dimension for plotting (1, 28, 28) -> (28, 28)
        img = images[i].squeeze().numpy()
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f'Label: {labels[i].item()}')
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

print("Training examples:")
visualize_dataset(train_loader)

In [ ]:
# Plot class distribution
def plot_class_distribution(datasets, dataset_names):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for idx, (dataset, name) in enumerate(zip(datasets, dataset_names)):
        if hasattr(dataset, 'targets'):
            labels = dataset.targets.numpy()
        else:
            labels = [dataset[i][1] for i in range(len(dataset))]
            labels = np.array(labels)

        unique, counts = np.unique(labels, return_counts=True)
        axes[idx].bar(unique, counts)
        axes[idx].set_title(f'{name} Set - Class Distribution')
        axes[idx].set_xlabel('Digit')
        axes[idx].set_ylabel('Count')
        axes[idx].set_xticks(range(10))

    plt.tight_layout()
    plt.show()

plot_class_distribution(
    [train_dataset, val_dataset, test_dataset],
    ['Train', 'Validation', 'Test']
)


In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(MNISTCNN, self).__init__()

        # First convolutional block
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Second convolutional block
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # Pooling layer
        self.pool = nn.MaxPool2d(2, 2)

        # Dropout for regularization
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)

        # Fully connected layers
        # After two pooling layers: 28x28 -> 14x14 -> 7x7
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # First conv block
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.dropout1(x)

        # Second conv block
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = self.dropout1(x)

        # Flatten
        x = x.view(-1, 128 * 7 * 7)

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)

        return x

# Create model
model = MNISTCNN(num_classes=10).to(device)
print(model)

# Print model summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {count_parameters(model):,}")

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=10):
    print("Starting training...")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (data, targets) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)

            outputs = model(data)
            loss = criterion(outputs, targets)

            # Backward pass and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for data, targets in val_loader:
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                loss = criterion(outputs, targets)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

        val_loss = val_loss / len(val_loader)
        val_acc = 100. * correct / total

        # Update learning rate
        scheduler.step()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, '
              f'LR: {scheduler.get_last_lr()[0]:.6f}')

    print("Training complete.")
    return train_losses, val_losses, train_accs, val_accs

# Train the model
train_losses, val_losses, train_accs, val_accs = train_model(
    model, train_loader, val_loader, num_epochs=15
)

In [ ]:

# %%
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(train_losses, label='Training Loss', linewidth=2)
ax1.plot(val_losses, label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(train_accs, label='Training Accuracy', linewidth=2)
ax2.plot(val_accs, label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = outputs.max(1)

            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    accuracy = 100. * correct / total
    print(f'Test Accuracy: {accuracy:.2f}%')

    return all_predictions, all_targets, accuracy

print("Evaluating on test set...")
test_predictions, test_targets, test_accuracy = evaluate_model(model, test_loader)

In [ ]:
# Confusion Matrix
def plot_confusion_matrix(predictions, targets):
    cm = confusion_matrix(targets, predictions)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=range(10), yticklabels=range(10))
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

print("\nClassification Report:")
print(classification_report(test_targets, test_predictions))

plot_confusion_matrix(test_predictions, test_targets)

In [ ]:
def visualize_predictions(model, test_loader, num_examples=15):
    model.eval()
    data_iter = iter(test_loader)
    images, labels = next(data_iter)

    images, labels = images.to(device), labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        _, predictions = outputs.max(1)
        probabilities = F.softmax(outputs, dim=1)
        confidence_scores, _ = probabilities.max(1)

    images = images.cpu().numpy()
    labels = labels.cpu().numpy()
    predictions = predictions.cpu().numpy()
    confidence_scores = confidence_scores.cpu().numpy()

    fig, axes = plt.subplots(3, 5, figsize=(15, 9))
    axes = axes.ravel()

    for i in range(num_examples):
        img = images[i].squeeze()

        axes[i].imshow(img, cmap='gray')

        # Color title based on correctness
        if labels[i] == predictions[i]:
            color = 'green'
            status = '✓'
        else:
            color = 'red'
            status = '✗'

        axes[i].set_title(f'True: {labels[i]}, Pred: {predictions[i]} {status}\nConf: {confidence_scores[i]:.3f}',
                         color=color, fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

print("Visualizing predictions...")
visualize_predictions(model, test_loader)

In [ ]:
def visualize_filters(model):
    conv1 = model.conv1
    filters = conv1.weight.data.cpu().numpy()

    # Plot first 16 filters
    fig, axes = plt.subplots(4, 8, figsize=(12, 6))
    for i in range(32):
        if i >= 32:
            break
        ax = axes[i//8, i%8]
        filter_img = filters[i, 0]  # First channel (grayscale)

        # Normalize for visualization
        filter_img = (filter_img - filter_img.min()) / (filter_img.max() - filter_img.min())

        ax.imshow(filter_img, cmap='viridis')
        ax.set_title(f'Filter {i+1}')
        ax.axis('off')

    plt.suptitle('First Convolutional Layer Filters')
    plt.tight_layout()
    plt.show()

def visualize_feature_maps(model, test_loader):
    model.eval()
    data_iter = iter(test_loader)
    images, labels = next(data_iter)

    image = images[0:1].to(device)

    feature_maps = []

    def hook_fn(module, input, output):
        feature_maps.append(output)

    # Register hook to first conv layer
    hook = model.conv1.register_forward_hook(hook_fn)

    with torch.no_grad():
        _ = model(image)

    hook.remove()

    # Visualize feature maps
    if feature_maps:
        maps = feature_maps[0].cpu().numpy()[0]  # First batch, first image

        # Plot first 16 feature maps
        fig, axes = plt.subplots(4, 8, figsize=(15, 8))
        for i in range(32):
            if i >= 32:
                break
            ax = axes[i//8, i%8]
            ax.imshow(maps[i], cmap='viridis')
            ax.set_title(f'Map {i+1}')
            ax.axis('off')

        plt.suptitle('Feature Maps from First Convolutional Layer')
        plt.tight_layout()
        plt.show()

print("Visualizing filters")
visualize_filters(model)

print("Visualizing feature maps")
visualize_feature_maps(model, test_loader)

DL is all about experiments. Make as many experiments with MLP on MNIST datasets and visualize your results. Things to explore:

1. Different architectures:
    * number of hidden layers
    * sizes of hidden layers
    * including/excluding dropout and/or batchnorm
    * activation function

2. Tuning parameter (such as learning rate) of different optimizers: try
    * SGD
    * Adam
    * RMSProp
    * etc
    
Do not forget to display test accuracy and training time of each model